In [67]:
import kagglehub
import pandas as pd
import os

# 1. Baixa o dataset e descobre o caminho real onde ele foi salvo no seu PC
path = kagglehub.dataset_download("namespaiva/base-varejo")

# 2. Junta o caminho da pasta com o nome do seu arquivo CSV

full_path = os.path.join(path, "Base Varejo.csv")

# 3. Lê o arquivo normalmente usando o Pandas
df = pd.read_csv(full_path, encoding='utf-8', sep=';')

#Criando dicionario para mapear os códigos de estado civil para seus significados de acordo com a descrição do dataset
dic_estado_civil = {1: "Casado/UE", 2: "Divorciado", 3: "Separado", 4: "Solteiro", 5: "Viúvo", 6: "Viúvo(a)"}

print("\nInformações do DataFrame:")
print(df.shape)
print(df.info())
print()
print("Primeiros 5 registros:")
print(df.head())



Informações do DataFrame:
(830000, 14)
<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  str    
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  str    
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  str    
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  str    
 9   PR_NOME      830000 non-null  str    
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), str(5)
memory usage: 88.7 MB
None

Primeiros 5 registros:
         DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID     PR_CAT  \
0  

# limpeza

### Limpando e convertendo nomes e tipos de Colunas

- Removendo colunas em branco "Unnamed: 10", "Unnamed: 11", "Unnamed: 12", "Unnamed: 13".
- Renomeando colunas de acordo com a descrição do dataset
- Convetendo data para datetime

In [68]:

# remove as colunas extras que não possuem nome
df = df.drop(columns=["Unnamed: 10", "Unnamed: 11", "Unnamed: 12", "Unnamed: 13"])

# renomeia as colunas para nomes mais amigáveis
df = df.rename(columns={"CO_ID": "N_fiscal", "CL_ID": "ID_Cliente", "CL_GENERO": "Genero", "CL_EC": "Estado_Civil", "CL_FHL": "Qtd_Filhos", "CL_SEG": "Seg_Economica", "PR_ID": "ID_Produto", "PR_CAT": "Cat_Produto",
                         "PR_NOME": "Nome_Produto"})
#converte a coluna "DATA" para o formato de datetime do Pandas
df["DATA"] = pd.to_datetime(
    df["DATA"],
    format="%d/%m/%Y",
    errors="coerce")


## Duplicatas

In [69]:
# Identificando duplicatas
dup_mask = df.duplicated()
print(f"Total de duplicatas: {dup_mask.sum()}")
print()

# # Exemplo das duplicatas encontradas
print("Primeiras 4 duplicatas:")
print(df[dup_mask].head(4).to_string()) 
#removendo as duplicatas
df_limpo = df.drop_duplicates()
print(f"Tamanho original: {len(df)} - Tamanho após remoção de duplicatas: {len(df_limpo)}, foram removidas {(len(df) - len(df_limpo))/len(df)*100:.2f}% de linhas totais que eram duplicadas.")


Total de duplicatas: 96553

Primeiras 4 duplicatas:
         DATA  N_fiscal  ID_Cliente Genero  Estado_Civil  Qtd_Filhos Seg_Economica  ID_Produto Cat_Produto        Nome_Produto
19 2019-02-01      1000         534      M             4           1             C          13   ALIMENTOS              BANANA
40 2019-02-01      1000         534      M             4           1             C           4   ALIMENTOS             ABACAXI
46 2019-02-01      1000         534      M             4           1             C          11   ALIMENTOS              AZEITE
49 2019-02-01      1000         534      M             4           1             C          69     BEBIDAS  REFRIGERANTE LIMaO
Tamanho original: 830000 - Tamanho após remoção de duplicatas: 733447, foram removidas 11.63% de linhas totais que eram duplicadas.


In [70]:
print(df_limpo.info())
print(df_limpo.head())

<class 'pandas.DataFrame'>
Index: 733447 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   DATA           733447 non-null  datetime64[us]
 1   N_fiscal       733447 non-null  int64         
 2   ID_Cliente     733447 non-null  int64         
 3   Genero         733447 non-null  str           
 4   Estado_Civil   733447 non-null  int64         
 5   Qtd_Filhos     733447 non-null  int64         
 6   Seg_Economica  733447 non-null  str           
 7   ID_Produto     733447 non-null  int64         
 8   Cat_Produto    733447 non-null  str           
 9   Nome_Produto   733447 non-null  str           
dtypes: datetime64[us](1), int64(5), str(4)
memory usage: 61.6 MB
None
        DATA  N_fiscal  ID_Cliente Genero  Estado_Civil  Qtd_Filhos  \
0 2019-02-01      1000         534      M             4           1   
1 2019-02-01      1000         534      M             4           1 

### Analisando dados

In [71]:
dic_estado_civil = {1: "Casado/UE", 2: "Divorciado", 3: "Separado", 4: "Solteiro", 5: "Viúvo", 6: "Viúvo(a)"}
print(df_limpo["Estado_Civil"].map(dic_estado_civil).value_counts())

Estado_Civil
Separado      189048
Solteiro      179206
Divorciado    172283
Casado/UE     172010
Viúvo          20900
Name: count, dtype: int64


In [72]:
colunas_categoricas = ["Genero", "Estado_Civil", "Qtd_Filhos", "Seg_Economica", "Cat_Produto"]
for col in colunas_categoricas:
    if col == "Estado_Civil":
        print(f"{(df[col].map(dic_estado_civil).value_counts()/len(df[col])*100).round(2)}%")
    else:
        print(f"\nContagem de valores para a coluna '{col}':")
        print(f"{(df[col].value_counts()/len(df[col])*100).round(2)}%")
        print("-"*20)



Contagem de valores para a coluna 'Genero':
Genero
F    52.12
M    47.88
Name: count, dtype: float64%
--------------------
Estado_Civil
Separado      25.75
Solteiro      24.41
Divorciado    23.49
Casado/UE     23.48
Viúvo          2.86
Name: count, dtype: float64%

Contagem de valores para a coluna 'Qtd_Filhos':
Qtd_Filhos
0    52.47
2    12.85
3    12.61
1    12.39
4     9.68
Name: count, dtype: float64%
--------------------

Contagem de valores para a coluna 'Seg_Economica':
Seg_Economica
B    63.88
C    27.96
A     8.16
Name: count, dtype: float64%
--------------------

Contagem de valores para a coluna 'Cat_Produto':
Cat_Produto
ALIMENTOS     52.38
HIGIENE       18.74
LIMPEZA       17.56
BEBIDAS        5.22
PET            3.90
ACESSORIOS     1.75
#N/D           0.44
Name: count, dtype: float64%
--------------------


Identificado Valor #N/D nas categorias do produto, verificando se existe algum padrão.

In [ ]:
df_analise = df_limpo.query("Cat_Produto == '#N/D'")
print(df_analise.info())
print(df_analise["ID_Produto"].value_counts())

ID_Produto
107    3228
Name: count, dtype: int64


In [63]:
df["Cat_Produto"] = df["Cat_Produto"].replace("#N/D", "Desconhecido")

In [64]:
print((df["Cat_Produto"].value_counts()/len(df["Cat_Produto"])*100).round(2))

Cat_Produto
ALIMENTOS       52.38
HIGIENE         18.74
LIMPEZA         17.56
BEBIDAS          5.22
PET              3.90
ACESSORIOS       1.75
Desconhecido     0.44
Name: count, dtype: float64


In [65]:
print(df.info())
print(df.describe().round(2))
print()
print("Valores duplicados por coluna:")
for column in df.columns:
    print(f"{column}: {df[column].duplicated().sum()}")
print()
print("Valores nulos por coluna:")
for column in df.columns:
    print(f"{column}: {df[column].isnull().sum()}")
print()
print(f"Valores únicos {column}: {df['Qtd_Filhos'].nunique()}")

<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   DATA           830000 non-null  datetime64[us]
 1   N_fiscal       830000 non-null  int64         
 2   ID_Cliente     830000 non-null  int64         
 3   Genero         830000 non-null  str           
 4   Estado_Civil   830000 non-null  int64         
 5   Qtd_Filhos     830000 non-null  int64         
 6   Seg_Economica  830000 non-null  str           
 7   ID_Produto     830000 non-null  int64         
 8   Cat_Produto    830000 non-null  str           
 9   Nome_Produto   830000 non-null  str           
dtypes: datetime64[us](1), int64(5), str(4)
memory usage: 63.3 MB
None
                             DATA   N_fiscal  ID_Cliente  Estado_Civil  \
count                      830000  830000.00   830000.00     830000.00   
mean   2020-12-06 14:55:46.421204  460045.09      499.60 